# AK-MCS basic example

This notebook runs a small AK-MCS example and visualizes the sampled population, initial DoE points, and adaptively added DoE points.


In [ ]:
import numpy as np
from ak_based_sampler import AK_MCS


def limit_state_function(u):
    return np.minimum.reduce([
        3 + 0.1 * (u[..., 0] - u[..., 1])**2 - (u[..., 0] + u[..., 1]) / np.sqrt(2),
        3 + 0.1 * (u[..., 0] - u[..., 1])**2 + (u[..., 0] + u[..., 1]) / np.sqrt(2),
        (u[..., 0] - u[..., 1]) + 6 / np.sqrt(2),
        (u[..., 1] - u[..., 0]) + 6 / np.sqrt(2),
    ])


SyntaxError: invalid syntax (2876811696.py, line 2)

In [ ]:
sampler = AK_MCS(dim=2, LSF=limit_state_function)

# Keep the example lightweight enough to run interactively.
sampler.set_parameters(
    n_initial_population=20_000,
    n_initial_DoEs=20,
    cov_stop=0.05,
    max_iterations_u=50,
    max_iterations_cov=20,
)

pf = sampler(seed=1, verbose=True)

print(f"Estimated failure probability: {pf:.6g}")
print(dict(sampler.result))


In [ ]:
import matplotlib.pyplot as plt

initial_u, initial_g = sampler.train_data["initial_DoE"]
added_u, added_g = sampler.train_data["DoE"]
population_u, population_g, population_sigma = sampler.train_data["population"]

failure_mask = population_g <= 0.0

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(
    population_u[~failure_mask, 0],
    population_u[~failure_mask, 1],
    s=4,
    c="#4c78a8",
    alpha=0.15,
    linewidths=0,
    label="Predicted safe population",
)
ax.scatter(
    population_u[failure_mask, 0],
    population_u[failure_mask, 1],
    s=4,
    c="#e45756",
    alpha=0.2,
    linewidths=0,
    label="Predicted failure population",
)
ax.scatter(
    initial_u[:, 0],
    initial_u[:, 1],
    s=60,
    c="#f2cf5b",
    edgecolors="black",
    linewidths=0.7,
    label="Initial DoE",
)
if len(added_u) > 0:
    ax.scatter(
        added_u[:, 0],
        added_u[:, 1],
        s=48,
        marker="x",
        c="#54a24b",
        linewidths=1.4,
        label="Added DoE",
    )

grid_x = np.linspace(population_u[:, 0].min(), population_u[:, 0].max(), 250)
grid_y = np.linspace(population_u[:, 1].min(), population_u[:, 1].max(), 250)
xx, yy = np.meshgrid(grid_x, grid_y)
zz = limit_state_function(np.stack([xx, yy], axis=-1))
ax.contour(xx, yy, zz, levels=[0.0], colors="black", linewidths=1.6)

ax.set_title("AK-MCS sampling result")
ax.set_xlabel("u1")
ax.set_ylabel("u2")
ax.set_aspect("equal", adjustable="box")
ax.legend(loc="upper right", frameon=True)
ax.grid(alpha=0.2)
plt.show()
